# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a step-by-step demonstration for loading and exploring the FAIR² dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains detailed clinicopathological and molecular features of 77 cancer survivors with second primary colorectal cancer.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Summarize metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Let's review the available record sets, fields, and their `@id`s. We'll inspect all top-level record sets defined in the Croissant schema and print their structure, always referencing entities by `@id`.

In [ ]:
# List all available record sets in the dataset, referencing by @id
record_set_objs = getattr(md, 'recordSet', [])
if not isinstance(record_set_objs, list):
    record_set_objs = [record_set_objs]
print('Available Record Sets:')
for rs in record_set_objs:
    print(f"  - {getattr(rs, '@id', None)}: {getattr(rs, 'name', 'Unnamed')}")

# Show details for each record set: list fields and their @ids
for rs in record_set_objs:
    print(f"\nRecord Set: {getattr(rs, 'name', 'Unnamed')} (@id: {getattr(rs, '@id', None)})")
    fields = getattr(rs, 'field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        print(f"    - {getattr(f, '@id', None)}: {getattr(f, 'name', 'Unnamed')}")

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis. We'll use the exact `@id` values from above to maintain clarity and reproducibility throughout.

In [ ]:
# Extract all record set IDs by @id
record_set_ids = []
for rs in record_set_objs:
    rs_id = getattr(rs, '@id', None)
    if rs_id:
        record_set_ids.append(rs_id)

# Load data for each record set into a dictionary of DataFrames
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[rs_id])} records from Record Set '@id': {rs_id}")
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")

# Show available columns/fields for each record set
for rs_id, df in dataframes.items():
    print(f"\nColumns in Record Set '@id': {rs_id}")
    print(df.columns.tolist())
    display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Let's apply some common data processing steps. We'll select a record set containing numeric clinical variables, filter on values, normalize data, and group by a key attribute. All fields are referenced by their `@id` as required.

In [ ]:
# Select a record set with numeric fields; customize the @id below as observed in Section 2
# Replace the following @ids with the actual ones found in your exploration above
# Example placeholder values (adjust as necessary):
main_record_set = record_set_ids[0] if record_set_ids else None
df = dataframes.get(main_record_set)

print(f"Analyzing Record Set: {main_record_set}")

# Inspect columns to choose numeric fields
print('Available columns:', df.columns.tolist() if df is not None else '(no dataframe)')

# Example: Let's try to select a field likely containing age or some numeric value.
# Substitute with the relevant field @id from the columns as needed
numeric_field = None
for col in df.columns:
    if 'age' in col.lower() or 'interval' in col.lower() or 'count' in col.lower() or 'number' in col.lower():
        numeric_field = col
        break
if not numeric_field:
    # fallback to the first numeric-looking column
    nums = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    numeric_field = nums[0] if nums else df.columns[0]

# Only proceed if the numeric field exists
if numeric_field:
    # Attempt converting to numeric in case values are stored as string
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].median()  # Example: use the median as a threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a category field
    group_field = None
    for col in df.columns:
        if col != numeric_field and (df[col].nunique() < 10):
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by '{group_field}':")
        display(grouped_df)
else:
    print('No suitable numeric field found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib/seaborn as a demonstration.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field
if df is not None and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If a grouping field was found, plot group mean
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"Mean '{numeric_field}' by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to programmatically load, overview, and process a Croissant-format dataset using `mlcroissant`. All entities were referenced by their unique `@id` values for clarity and reproducibility. The workflow included EDA and visualization of main numeric features, grouped by key attributes, on the FAIR² colorectal cancer survivor dataset.